# 04 · Data integration, quality control and statistical validation
**Owner:** Chan Jia Yi · **Run this notebook first.** It builds the shared data every other notebook reads.

| Section | What it does | Output |
|---|---|---|
| 1. Integration | Raw CSV → one Parquet with all 10M rows + shared derived columns; sample and anomaly extracts | `data/processed/*.parquet` |
| 2. Quality control | One issues log: what was checked, what was found, the decision and why | `outputs/tables/qc_*.csv` |
| 3. Statistical validation | Tests every Currency / Amount / Business-consumer rule, with effect sizes and a verdict | `outputs/tables/rules/rules_04_core.csv` |

**Definitions used by all notebooks**
- *Country* = the **bank (BIC) country**, read from BIC characters 5–6. It is not a verified customer residence.
- *Local currency* = the currency a country's senders use most, **inferred from this dataset** (no external lookup).
- Nothing is deleted. Anomalies and outliers are kept and flagged. `payments_clean` means "parsed and typed", not "every row is valid".
- Verdict scale: **Confirmed** (≥99% of rows, or the effect matches the rule) · **Strong** (95–99%) · **Partial** (70–95%, or only in a subgroup) · **Rejected**.

In [1]:
from pathlib import Path
import os, warnings; warnings.filterwarnings('ignore')
import duckdb, numpy as np, pandas as pd
import matplotlib.pyplot as plt
pd.set_option('display.width', 220); pd.set_option('display.max_columns', 60)

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
RAW, PROC = ROOT / 'data' / 'raw', ROOT / 'data' / 'processed'
FIG, TAB = ROOT / 'outputs' / 'figures', ROOT / 'outputs' / 'tables'
RULES = TAB / 'rules'                       # one rules file per notebook -> combined into outputs/tables/rules_log.csv
for d in (RAW, PROC, FIG, TAB, RULES): d.mkdir(parents=True, exist_ok=True)
FULL    = PROC / 'payments_clean.parquet'   # all 10M rows (nothing removed) + derived columns, built by notebook 04
SAMPLE  = PROC / 'sample_20pct.parquet'     # 20% Bernoulli sample (seed 42), no free-text columns, still includes anomalies
ANOM    = PROC / 'anomalies.parquet'        # every labelled anomaly row, all columns
SCHEMA_VERSION = '2'                        # bump when derived columns change -> notebook 04 rebuilds the files

def connect(db=':memory:'):
    con = duckdb.connect(str(db))
    con.execute("SET TimeZone='UTC'")       # never skip: otherwise DuckDB shows laptop time (+08)
    con.execute("SET preserve_insertion_order=false")
    try: con.execute("SET enable_progress_bar=false")   # no progress widgets saved in the notebook ("Could not render ... widget" in viewers)
    except Exception: pass                           # (older setups without ipywidgets show no widget anyway)
    con.execute(f"SET temp_directory='{(PROC / 'duck_tmp').as_posix()}'")   # big joins spill to disk instead of crashing
    if os.environ.get('SWIFT_MEMORY'):  con.execute(f"SET memory_limit='{os.environ['SWIFT_MEMORY']}'")   # e.g. 4GB on an 8 GB laptop
    if os.environ.get('SWIFT_THREADS'): con.execute(f"SET threads={int(os.environ['SWIFT_THREADS'])}")
    return con

In [2]:
con = connect()

In [3]:
def sql(q):
    """Run SQL on the full data and return a pandas DataFrame."""
    return con.execute(q).df()
q = sql
def save_table(df, name):
    """Save a result table to outputs/tables/<name>.csv and return it for display."""
    df.to_csv(TAB / f'{name}.csv', index=False); return df

RULE_COLS = ['id', 'group', 'rule', 'test', 'value', 'verdict', 'evidence_files', 'interpretation']
VERDICTS = 'Confirmed (holds in >=99% of rows / effect matches) · Strong (95-99%) · Partial (70-95% or only in a subgroup) · Rejected'
def verdict_from_share(x):
    return 'Confirmed' if x >= 0.99 else 'Strong' if x >= 0.95 else 'Partial' if x >= 0.70 else 'Rejected'

def combine_rules():
    """Merge every outputs/tables/rules/rules_*.csv into one outputs/tables/rules_log.csv (shared schema)."""
    frames = []
    for f in sorted(RULES.glob('rules_*.csv')):
        d = pd.read_csv(f)
        ren = {'observed_evidence_table_or_chart': 'evidence_files', 'interpretation_real_world': 'interpretation', 'confidence': 'verdict'}
        d = d.rename(columns={k: v for k, v in ren.items() if k in d.columns and v not in d.columns})
        if 'rule' not in d.columns and {'when_condition', 'then_effect'} <= set(d.columns):
            d['rule'] = d.when_condition.astype(str) + ' -> ' + d.then_effect.astype(str)
        if 'value' not in d.columns and 'size' in d.columns: d['value'] = d['size']
        d['source'] = f.name; frames.append(d)
    if not frames: return pd.DataFrame(columns=RULE_COLS)
    out = pd.concat(frames, ignore_index=True)
    out = out[[c for c in RULE_COLS + ['source'] if c in out.columns] + [c for c in out.columns if c not in RULE_COLS + ['source']]]
    out.to_csv(TAB / 'rules_log.csv', index=False); return out

## 1. Data integration
- The raw file is ~5 GB, too large for Excel or plain pandas. DuckDB streams it once into a compressed Parquet (~1.3 GB).
- The text columns contain line breaks inside quotes, so the CSV is read with **explicit quoting and fixed column types** (if types were guessed, one bad row could silently turn a number column into text). If the fast parallel reader fails, it retries single-threaded.
- `DISCLAIMER` (one non-empty value) is printed, then dropped.
- **Derived columns are added once here**, so every notebook uses the same definitions. The first build takes about 5–10 minutes. Later runs reuse the files unless `SCHEMA_VERSION` changes.

In [4]:
COLS = {
    "DISCLAIMER": "VARCHAR", "EPOCH_SECONDS": "BIGINT", "CURRENCY_CODE": "VARCHAR", "USD_AMOUNT": "DOUBLE",
    "ORDERING_BIC8": "VARCHAR", "BENEFICIARY_BIC8": "VARCHAR", "BIC8_PATH": "VARCHAR", "TOTAL_DURATION": "BIGINT",
    "CONTINUITY_QFLAG": "VARCHAR", "CURRENCY_CONVERSION_COUNT": "INTEGER", "PARTICIPANT_COUNT": "INTEGER",
    "TRANSACTION_TYPE_CODE": "VARCHAR", "ORDERING_ACCOUNT": "VARCHAR", "BENEFICIARY_ACCOUNT": "VARCHAR",
    "ORDERING_DETAILS": "VARCHAR", "ORDERING_TOKENS": "VARCHAR", "BENEFICIARY_DETAILS": "VARCHAR",
    "BENEFICIARY_TOKENS": "VARCHAR", "ORDERING_TYPE": "VARCHAR", "BENEFICIARY_TYPE": "VARCHAR", "ANOMALY_TYPE": "VARCHAR",
}
DELIM = ','                                   # change to '\t' only if the file is tab-separated
csvs = sorted(RAW.glob('*.csv'))
assert len(csvs) == 1, f'Put exactly one competition CSV in data/raw/ (found {len(csvs)})'
SRC = csvs[0]; print('raw file:', SRC.name, f'({SRC.stat().st_size/1e9:.2f} GB)')

def csv_source(parallel=True):
    cols = ', '.join(f"'{k}': '{v}'" for k, v in COLS.items())
    return (f"read_csv('{SRC.as_posix()}', header=true, delim='{DELIM}', quote='\"', escape='\"', "
            f"columns={{{cols}}}, max_line_size=10000000, parallel={str(parallel).lower()})")

DERIVED = r"""
  SELECT * EXCLUDE (DISCLAIMER),
    row_number() OVER ()                                          AS rid,          -- stable row id (joins between notebooks)
    to_timestamp(EPOCH_SECONDS)                                   AS ts,           -- UTC
    substr(ORDERING_BIC8, 5, 2)                                   AS o_cc,         -- sender bank country
    substr(BENEFICIARY_BIC8, 5, 2)                                AS b_cc,         -- receiver bank country
    ORDERING_TYPE || '2' || BENEFICIARY_TYPE                      AS pair,         -- B2B / B2C / C2B / C2C
    coalesce(nullif(trim(ANOMALY_TYPE), ''), 'NONE')              AS anomaly,      -- blank label = normal payment
    substr(ORDERING_BIC8, 5, 2) <> substr(BENEFICIARY_BIC8, 5, 2) AS cross_border,
    len(string_split(BIC8_PATH, ';'))                             AS path_len,
    contains(BIC8_PATH, '...')                                    AS path_gap,
    regexp_extract(ORDERING_TOKENS,    '<country_(?:code|name):([A-Z]{2})', 1) AS o_det_cc,   -- country written in the details
    regexp_extract(BENEFICIARY_TOKENS, '<country_(?:code|name):([A-Z]{2})', 1) AS b_det_cc,
    trim(regexp_extract(replace(ORDERING_DETAILS,    chr(13), ''), '^/+([^\n]*)', 1)) AS o_det_acct, -- account line in the details
    trim(regexp_extract(replace(BENEFICIARY_DETAILS, chr(13), ''), '^/+([^\n]*)', 1)) AS b_det_acct,
    regexp_replace(replace(BENEFICIARY_DETAILS, chr(13), ''), '^/[^\n]*\n?', '')      AS b_party_text -- details minus account line
  FROM __SRC__"""

ver_file = PROC / '_schema_version.txt'
if not (FULL.exists() and ver_file.exists() and ver_file.read_text().strip() == SCHEMA_VERSION):
    for f in (FULL, SAMPLE, ANOM): f.unlink(missing_ok=True)          # outdated derived files are rebuilt together
    for parallel in (True, False):
        try:
            src = csv_source(parallel)
            print('DISCLAIMER:', con.execute(f'SELECT DISCLAIMER FROM {src} WHERE DISCLAIMER IS NOT NULL LIMIT 1').fetchone()[0][:400])
            con.execute(f"COPY ({DERIVED.replace('__SRC__', src)}) TO '{FULL.as_posix()}' (FORMAT parquet, COMPRESSION zstd)")
            print('Parquet written, parallel =', parallel); break
        except Exception as e:
            print('Load failed with parallel =', parallel, '->', str(e)[:300])
    else:
        raise RuntimeError('CSV could not be parsed. Check DELIM and the file in data/raw/.')
    ver_file.write_text(SCHEMA_VERSION)
print('shared parquet:', FULL.name, f'({FULL.stat().st_size/1e9:.2f} GB)')

raw file: swift_umdac_datathon_2026_fictitious_payment_data.csv (4.95 GB)
shared parquet: payments_clean.parquet (1.71 GB)


In [5]:
# Main view `p` = full data with the shared derived columns (see notebook 04, section 1)
assert FULL.exists() and (PROC / '_schema_version.txt').exists() and (PROC / '_schema_version.txt').read_text().strip() == SCHEMA_VERSION, \
    'Shared data missing or outdated: run notebooks/04_integration_validation.ipynb first'
con.execute(f"CREATE OR REPLACE VIEW p AS SELECT * FROM read_parquet('{FULL.as_posix()}')")
# Local currency of each BIC country = the currency its senders use most (inferred from this dataset, no external lookup)
con.execute("""CREATE OR REPLACE TABLE local_ccy AS
  SELECT o_cc AS cc, arg_max(CURRENCY_CODE, n) AS local_ccy FROM
  (SELECT o_cc, CURRENCY_CODE, count(*) n FROM p GROUP BY ALL) GROUP BY o_cc""")
print(f"{con.execute('SELECT count(*) FROM p').fetchone()[0]:,} payments loaded")

10,000,000 payments loaded


In [6]:
# Parse check: numeric columns must have no NULLs (a bad parse shows up here first)
chk = sql("""SELECT count(*) AS rows, count(EPOCH_SECONDS) AS epoch_ok, count(USD_AMOUNT) AS amount_ok,
  min(ts) AS first_ts, max(ts) AS last_ts, count(DISTINCT CURRENCY_CODE) AS currencies, count(DISTINCT o_cc) AS sender_countries,
  sum((anomaly <> 'NONE')::INT) AS anomaly_rows FROM p""")
assert chk.loc[0, 'rows'] == chk.loc[0, 'epoch_ok'] == chk.loc[0, 'amount_ok'], 'Some rows did not parse: check DELIM / quoting'
display(chk)

# Extracts for the analysis notebooks
if not SAMPLE.exists():
    con.execute(f"""COPY (SELECT * EXCLUDE (ORDERING_DETAILS, ORDERING_TOKENS, BENEFICIARY_DETAILS, BENEFICIARY_TOKENS, b_party_text)
                    FROM p USING SAMPLE 20 PERCENT (bernoulli, 42)) TO '{SAMPLE.as_posix()}' (FORMAT parquet, COMPRESSION zstd)""")
if not ANOM.exists():
    con.execute(f"COPY (SELECT * FROM p WHERE anomaly <> 'NONE') TO '{ANOM.as_posix()}' (FORMAT parquet, COMPRESSION zstd)")
for f in (SAMPLE, ANOM):
    rows = con.execute("SELECT count(*) FROM read_parquet('" + f.as_posix() + "')").fetchone()[0]
    print(f.name, f'{rows:,} rows')
sql('DESCRIBE p')[['column_name', 'column_type']].T

,rows,epoch_ok,amount_ok,first_ts,last_ts,currencies,sender_countries,anomaly_rows
0,10000000,10000000,10000000,2023-04-30 11:22:26+00:00,2023-07-31 09:56:01+00:00,155,228,7006.0


sample_20pct.parquet 2,001,857 rows
anomalies.parquet 7,006 rows


,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32,33
column_name,EPOCH_SECONDS,CURRENCY_CODE,USD_AMOUNT,ORDERING_BIC8,BENEFICIARY_BIC8,BIC8_PATH,TOTAL_DURATION,CONTINUITY_QFLAG,CURRENCY_CONVERSION_COUNT,PARTICIPANT_COUNT,TRANSACTION_TYPE_CODE,ORDERING_ACCOUNT,BENEFICIARY_ACCOUNT,ORDERING_DETAILS,ORDERING_TOKENS,BENEFICIARY_DETAILS,BENEFICIARY_TOKENS,ORDERING_TYPE,BENEFICIARY_TYPE,ANOMALY_TYPE,rid,ts,o_cc,b_cc,pair,anomaly,cross_border,path_len,path_gap,o_det_cc,b_det_cc,o_det_acct,b_det_acct,b_party_text
column_type,BIGINT,VARCHAR,DOUBLE,VARCHAR,VARCHAR,VARCHAR,BIGINT,VARCHAR,INTEGER,INTEGER,VARCHAR,VARCHAR,VARCHAR,VARCHAR,VARCHAR,VARCHAR,VARCHAR,VARCHAR,VARCHAR,VARCHAR,BIGINT,TIMESTAMP WITH TIME ZONE,VARCHAR,VARCHAR,VARCHAR,VARCHAR,BOOLEAN,BIGINT,BOOLEAN,VARCHAR,VARCHAR,VARCHAR,VARCHAR,VARCHAR


## 2. Quality control → issues log
One row per check, with what was found, the decision and the reason. This table goes on the QC slide almost as-is.

In [7]:
display(save_table(sql("SUMMARIZE SELECT * EXCLUDE (ORDERING_DETAILS, BENEFICIARY_DETAILS, b_party_text) FROM p"), 'qc_summarize_columns'))

,column_name,column_type,min,max,approx_unique,avg,std,q25,q50,q75,count,null_percentage
0,EPOCH_SECONDS,BIGINT,1682853746,1690797361,5119419,1686927373.7817295,2191913.6682333206,1685171116,1687041094,1688640428,10000000,0.00
1,CURRENCY_CODE,VARCHAR,AED,ZWL,151,None,None,None,None,None,10000000,0.00
2,USD_AMOUNT,DOUBLE,0.01,15664132213.09,3441855,97204.1020382095,9909715.280074298,636.3949928076566,3675.613957340253,19992.38371132207,10000000,0.00
3,ORDERING_BIC8,VARCHAR,0000ADA0,1136US30,8709,None,None,None,None,None,10000000,0.00
4,BENEFICIARY_BIC8,VARCHAR,0000ADA0,1136US30,8632,None,None,None,None,None,10000000,0.00
5,BIC8_PATH,VARCHAR,0000ADA0;...;...;0000NL20;0023ILI0;0001MQM0,1136US30;0313FRP0;0006LV20,4122510,None,None,None,None,None,10000000,0.00
6,TOTAL_DURATION,BIGINT,2,21649,9468,771.5102084,1228.2022156000373,26,57,1109,10000000,0.00
7,CONTINUITY_QFLAG,VARCHAR,N,Y,2,None,None,None,None,None,10000000,0.00
8,CURRENCY_CONVERSION_COUNT,INTEGER,0,17,18,1.3430237,0.9446476223831941,1,1,2,10000000,0.00
9,PARTICIPANT_COUNT,INTEGER,2,22,19,2.5815715,0.9592645007538823,2,2,3,10000000,0.00


In [8]:
APPENDIX_LABELS = ['ROMANCE_SCAM', 'CEO_SCAM', 'ACCOUNT_REPLACEMENT_SCAM', 'FAT_FINGER_MISTAKE', 'DUPLICATE_PAYMENT']
APPX_SQL = ', '.join(f"'{a}'" for a in APPENDIX_LABELS)
RAWCOLS = ', '.join(c for c in COLS if c != 'DISCLAIMER')
checks = [   # (check, sql returning one value, pass condition on the value, decision, reason)
 ('Total rows', 'SELECT count(*) FROM p', lambda v: v > 0, 'Keep all', 'Baseline count'),
 ('Missing values in any column except ANOMALY_TYPE', 'SELECT ' + ' + '.join(f'count(*) - count("{c}")' for c in COLS if c not in ('DISCLAIMER', 'ANOMALY_TYPE')) + ' FROM p',
      lambda v: v == 0, 'None to fix', 'Every field is filled'),
 ('Normal rows (blank ANOMALY_TYPE)', "SELECT sum((anomaly='NONE')::INT) FROM p", lambda v: True, "Recoded blank to 'NONE'", 'Blank is a label, not missing data'),
 ('Exact duplicate rows (hash of all raw columns)', f'SELECT count(*) - count(DISTINCT hash({RAWCOLS})) FROM p', lambda v: v == 0,
      'Keep', 'DUPLICATE_PAYMENT is a labelled behaviour (see notebook 03, 4.3)'),
 ('Payments sharing time + accounts + amount + currency', """SELECT coalesce(sum(c), 0) FROM (SELECT count(*) c FROM p
      GROUP BY EPOCH_SECONDS, ORDERING_ACCOUNT, BENEFICIARY_ACCOUNT, USD_AMOUNT, CURRENCY_CODE HAVING count(*) > 1)""", lambda v: v == 0,
      'Keep, review', 'Same 5-field signature; not full-row copies'),
 ('USD_AMOUNT <= 0', 'SELECT sum((USD_AMOUNT <= 0)::INT) FROM p', lambda v: v == 0, 'Flag only', 'Would break log-amount analysis'),
 ('USD_AMOUNT range', "SELECT printf('%.2f to %,.0f', min(USD_AMOUNT), max(USD_AMOUNT)) FROM p", lambda v: True, 'Keep', 'Range check'),
 ('Label of the largest amount', 'SELECT arg_max(anomaly, USD_AMOUNT) FROM p', lambda v: v != 'NONE', '-', 'Extremes should be labelled errors'),
 ('TOTAL_DURATION < 0', 'SELECT sum((TOTAL_DURATION < 0)::INT) FROM p', lambda v: v == 0, 'Flag only', 'Impossible value'),
 ('Timestamp range (UTC)', "SELECT strftime(min(ts), '%Y-%m-%d %H:%M') || ' to ' || strftime(max(ts), '%Y-%m-%d %H:%M') FROM p", lambda v: True, '-', 'Coverage'),
 ('Currency code not 3 capital letters', "SELECT sum((NOT regexp_full_match(CURRENCY_CODE, '[A-Z]{3}'))::INT) FROM p", lambda v: v == 0, 'Flag only', 'Format check'),
 ('BIC8 not 8 characters', 'SELECT sum((length(ORDERING_BIC8) <> 8 OR length(BENEFICIARY_BIC8) <> 8)::INT) FROM p', lambda v: v == 0, 'Flag only', 'Country is read from BIC chars 5-6'),
 ('Party type not B/C', "SELECT sum((ORDERING_TYPE NOT IN ('B','C') OR BENEFICIARY_TYPE NOT IN ('B','C'))::INT) FROM p", lambda v: v == 0, 'Flag only', 'Label check'),
 ("CONTINUITY_QFLAG = N agrees with '...' in path (%)", "SELECT round(avg(((CONTINUITY_QFLAG='N') = path_gap)::INT)*100, 3) FROM p", lambda v: v == 100, '-', 'Appendix definition check'),
 ('Conversions > participants - 1', 'SELECT sum((CURRENCY_CONVERSION_COUNT > PARTICIPANT_COUNT - 1)::INT) FROM p', lambda v: v == 0, 'Flag only', 'Logical consistency'),
 ('Direct with PARTICIPANT_COUNT <> 2', "SELECT sum((TRANSACTION_TYPE_CODE='Direct' AND PARTICIPANT_COUNT<>2)::INT) FROM p", lambda v: v == 0, 'Flag only', 'Definition check'),
 ('ORDERING_BIC8 <> first BIC in path (%)', "SELECT round(avg((ORDERING_BIC8 <> split_part(BIC8_PATH, ';', 1))::INT)*100, 3) FROM p", lambda v: True, 'Keep; routing teammate', 'Possible routing rule'),
 ('BENEFICIARY_BIC8 <> last BIC in path (%)', "SELECT round(avg((BENEFICIARY_BIC8 <> list_last(string_split(BIC8_PATH, ';')))::INT)*100, 3) FROM p", lambda v: True, 'Keep; routing teammate', 'Possible routing rule'),
 ('Anomaly labels not in the appendix', f"SELECT coalesce(string_agg(DISTINCT anomaly, ', '), 'none') FROM p WHERE anomaly NOT IN ({APPX_SQL}, 'NONE')", lambda v: v == 'none', 'Keep; investigate (notebook 03, 4.7)', 'Undocumented label'),
 ('Rows with undocumented labels', f"SELECT count(*) FROM p WHERE anomaly NOT IN ({APPX_SQL}, 'NONE')", lambda v: v == 0, 'Keep; ask mentors', 'Undocumented label'),
 ('Details containing carriage return (\\r)', "SELECT sum((ORDERING_DETAILS LIKE '%' || chr(13) || '%' OR BENEFICIARY_DETAILS LIKE '%' || chr(13) || '%')::INT) FROM p",
      lambda v: True, 'Strip \\r before line-length checks', 'Would inflate line length by 1'),
]
rows = []
for c, s, ok, d, r in checks:
    v = con.execute(s).fetchone()[0]
    rows.append((c, v, 'pass' if ok(v) else 'REVIEW', d, r))
qc = save_table(pd.DataFrame(rows, columns=['check', 'result', 'status', 'decision', 'reason']), 'qc_issues_log')
anom = save_table(sql("SELECT anomaly, count(*) n, round(count(*) * 100.0 / sum(count(*)) OVER (), 4) AS pct FROM p GROUP BY 1 ORDER BY 2 DESC"), 'qc_anomaly_counts')
display(qc); display(anom)

,check,result,status,decision,reason
0,Total rows,10000000,pass,Keep all,Baseline count
1,Missing values in any column except ANOMALY_TYPE,0,pass,None to fix,Every field is filled
2,Normal rows (blank ANOMALY_TYPE),9992994,pass,Recoded blank to 'NONE',"Blank is a label, not missing data"
3,Exact duplicate rows (hash of all raw columns),1,REVIEW,Keep,DUPLICATE_PAYMENT is a labelled behaviour (see...
4,Payments sharing time + accounts + amount + cu...,2,REVIEW,"Keep, review",Same 5-field signature; not full-row copies
5,USD_AMOUNT <= 0,0,pass,Flag only,Would break log-amount analysis
6,USD_AMOUNT range,"0.01 to 15,664,132,213",pass,Keep,Range check
7,Label of the largest amount,FAT_FINGER_MISTAKE,pass,-,Extremes should be labelled errors
8,TOTAL_DURATION < 0,0,pass,Flag only,Impossible value
9,Timestamp range (UTC),2023-04-30 11:22 to 2023-07-31 09:56,pass,-,Coverage


,anomaly,n,pct
0,NONE,9992994,99.9299
1,ROMANCE_SCAM,2476,0.0248
2,FAT_FINGER_MISTAKE,1940,0.0194
3,CEO_SCAM,945,0.0095
4,DUPLICATE_PAYMENT,679,0.0068
5,ACCOUNT_REPLACEMENT_SCAM,649,0.0065
6,CUSTOMER_RECALL,317,0.0032


**QC interpretation**
- No row needs to be removed. Every anomaly row stays in the data with its label.
- `CUSTOMER_RECALL` is in the data but not in the official appendix. Ask the mentors on Discord. Notebook 03 (4.7) links these rows to earlier payments.
- `DUPLICATE_PAYMENT` rows are **not** full-row copies, so duplicates are matched by account, amount and time instead (notebook 03, 4.3).
- Passing format and range checks does not make every row realistic. Suspicious rows are analysed, not deleted.

## 3. Statistical validation of the Currency, Amount and Business/consumer rules
**Method.** Each rule is written as a testable statement, then measured on the **full data** (counts and accuracy) or on **normal payments in the 20% sample** (models).
- With 10M rows almost every p-value is ~0, so we report **effect sizes**: accuracy %, ln-effects with 95% intervals, R² and Cramér's V.
- Confounding is controlled by comparing payments **within the same sender country** (country fixed effect).
- The interval is a **cluster bootstrap**: it resamples whole sender accounts, not single payments, because payments from one account are not independent.

In [9]:
# Modelling sample: normal (non-anomaly) payments from the 20% sample
n = pd.read_parquet(SAMPLE, columns=['EPOCH_SECONDS','CURRENCY_CODE','USD_AMOUNT','o_cc','b_cc','ORDERING_TYPE','BENEFICIARY_TYPE',
        'pair','anomaly','ORDERING_BIC8','ORDERING_ACCOUNT','TRANSACTION_TYPE_CODE','CURRENCY_CONVERSION_COUNT','PARTICIPANT_COUNT'])
n = n[n.anomaly == 'NONE'].reset_index(drop=True)
n['la'] = np.log(n.USD_AMOUNT)                                   # ln(amount)
n['ob'] = (n.ORDERING_TYPE == 'B').astype(float)                 # sender is business
n['bb'] = (n.BENEFICIARY_TYPE == 'B').astype(float)              # receiver is business
loc = sql('SELECT * FROM local_ccy').set_index('cc').local_ccy
print(f'{len(n):,} normal payments in the modelling sample')

2,000,487 normal payments in the modelling sample


In [10]:
import pytz
TZMAP = {cc: z[0] for cc, z in pytz.country_timezones.items()}   # first zone for multi-zone countries (limitation)
def add_local_time(df, cc_col='o_cc'):
    """Adds local hour (lh) and weekday (ld, 0=Mon) of the sender BIC country. -1 if the country has no known zone."""
    ts = pd.to_datetime(df['EPOCH_SECONDS'], unit='s', utc=True)
    lh = np.full(len(df), -1); ld = np.full(len(df), -1)
    for z, idx in df.groupby(df[cc_col].map(TZMAP)).indices.items():
        t = ts.iloc[idx].dt.tz_convert(z); lh[idx] = t.dt.hour.values; ld[idx] = t.dt.dayofweek.values
    return df.assign(lh=lh, ld=ld)

In [11]:
from scipy import stats
def cramers_v(a, b):
    ct = pd.crosstab(a, b); chi2, p, _, _ = stats.chi2_contingency(ct)
    return np.sqrt(chi2 / (ct.values.sum() * (min(ct.shape) - 1))), p
def r2_by(y, g):
    return 1 - ((y - y.groupby(g).transform('mean')) ** 2).mean() / y.var()
def within(df, cols, by='o_cc'):          # demean inside each sender country (country fixed effect)
    return df[cols] - df.groupby(by)[cols].transform('mean')

W = within(n, ['la', 'ob', 'bb'])
X, y = W[['ob', 'bb']].values, W.la.values
beta = np.linalg.lstsq(X, y, rcond=None)[0]
resid = W.la - X @ beta

# Cluster bootstrap over sender accounts (Poisson weights per account; per-account sums make it fast)
acc = pd.factorize(n.ORDERING_BIC8 + '|' + n.ORDERING_ACCOUNT)[0]; K = acc.max() + 1
sxx = np.stack([np.bincount(acc, X[:, i] * X[:, j], K) for i, j in [(0, 0), (0, 1), (1, 1)]], 1)
sxy = np.stack([np.bincount(acc, X[:, i] * y, K) for i in (0, 1)], 1)
rng = np.random.default_rng(0); boot = []
for _ in range(300):
    w = rng.poisson(1.0, K); a, b, c = w @ sxx; d = w @ sxy
    boot.append(np.linalg.solve([[a, b], [b, c]], d))
ci = np.percentile(np.array(boot), [2.5, 97.5], axis=0)
print(f'{K:,} sender accounts; ln effect of business sender / receiver:', beta.round(3), '95% cluster CI:', ci.T.round(3).tolist())

209,274 sender accounts; ln effect of business sender / receiver: [1.999 2.002] 95% cluster CI: [[1.98, 2.019], [1.994, 2.009]]


In [12]:
log = []
def add(rid, group, rule, test, value, verdict, evidence, meaning):
    log.append(dict(id=rid, group=group, rule=rule, test=test, value=value, verdict=verdict, evidence_files=evidence, interpretation=meaning))

# ---- Currency (full data)
c1 = sql("""SELECT avg((CURRENCY_CODE = ls.local_ccy)::INT) sender,
                   avg((CURRENCY_CODE <> ls.local_ccy AND CURRENCY_CODE = lr.local_ccy)::INT) receiver_only,
                   avg((CURRENCY_CODE = lr.local_ccy)::INT) receiver_any
            FROM p LEFT JOIN local_ccy ls ON p.o_cc = ls.cc LEFT JOIN local_ccy lr ON p.b_cc = lr.cc""").iloc[0]
add('R-CCY1', 'Currency', "Payments are made in the sender country's local currency", 'full data; local currency inferred in-sample',
    f'{c1.sender:.1%} sender-local; only {c1.receiver_only:.1%} use the receiver currency instead ({c1.receiver_any:.1%} match the receiver currency incl. same-currency pairs)',
    verdict_from_share(c1.sender) + ' (exceptions not yet explained)', 'ccy1_currency_choice.png',
    "The sender's bank debits in its home currency; FX happens later along the route")
c2 = sql("""SELECT avg((CURRENCY_CONVERSION_COUNT = (ls.local_ccy <> lr.local_ccy)::INT)::INT) acc
            FROM p LEFT JOIN local_ccy ls ON p.o_cc = ls.cc LEFT JOIN local_ccy lr ON p.b_cc = lr.cc
            WHERE TRANSACTION_TYPE_CODE = 'Direct'""").iloc[0, 0]
add('R-CCY2', 'Currency', 'Direct payments: 1 conversion if sender and receiver local currencies differ, otherwise 0', 'prediction accuracy on all Direct rows',
    f'{c2:.1%}', verdict_from_share(c2), 'ccy2_direct_conversion_rule.png', 'With only two banks, FX happens once, and only between currency areas')
c3 = sql('SELECT avg((CURRENCY_CONVERSION_COUNT <= PARTICIPANT_COUNT - 1)::INT) FROM p').iloc[0, 0]
add('R-CCY3', 'Currency', 'Conversions never exceed the number of hops (participants - 1)', 'share of all rows', f'{c3:.2%}', verdict_from_share(c3),
    'ccy3_conversions_by_hops.csv', 'Each correspondent bank can convert at most once')

# ---- Amount (sample, normal payments)
add('R-AMT1', 'Amount', 'Amounts are log-normal: ln(amount) is normal with the same spread for every pair', 'skew / excess kurtosis / sd of ln residual',
    f'skew {stats.skew(resid):.3f}, kurtosis {stats.kurtosis(resid):.3f}, sd {resid.std():.2f}',
    'Confirmed' if abs(stats.skew(resid)) < 0.1 and abs(stats.kurtosis(resid)) < 0.2 else 'Partial', 'amt1_lognormal_by_pair.png',
    'Payment sizes built from many multiplying factors are naturally log-normal')
ok2 = (ci[0] <= 2.0).all() and (ci[1] >= 2.0).all() or (abs(beta - 2) < 0.05).all()
add('R-AMT2', 'Amount', 'Each business party (sender or receiver) multiplies the amount by e^2 = 7.4', 'ln effect within sender country, 95% cluster-bootstrap CI',
    f'sender {beta[0]:.3f} [{ci[0,0]:.3f}, {ci[1,0]:.3f}]; receiver {beta[1]:.3f} [{ci[0,1]:.3f}, {ci[1,1]:.3f}]', 'Confirmed' if ok2 else 'Partial',
    'amt2_business_multiplier.png', 'Businesses pay invoices and payroll; consumers send small personal transfers')
r2s, r2r = r2_by(n.la, n.o_cc), r2_by(pd.Series(resid), n.b_cc)
add('R-AMT3', 'Amount', "The sender's country sets the base amount; the receiver's country adds nothing", 'R² sender country vs extra R² of receiver country',
    f'{r2s:.3f} vs {r2r:.4f}', 'Confirmed' if r2s > 0.05 and r2r < 0.01 else 'Partial', 'amt3_country_base_amount.png',
    'Payment size scales with the income level of the sending economy (compare with GDP per capita, cited as a reference)')

# ---- Business / consumer
v, pv = cramers_v(n.ORDERING_TYPE, n.BENEFICIARY_TYPE)
add('R-BC1', 'Business/consumer', 'Sender type and receiver type are assigned independently', "chi-square p, Cramér's V",
    f'p = {pv:.2f}, V = {v:.4f}', 'Confirmed' if v < 0.01 else 'Rejected', 'bc1_type_independence.png', 'Who pays and who gets paid are separate draws')
lt = add_local_time(n.sample(min(500_000, len(n)), random_state=1)); lt = lt[lt.lh >= 0]
utc_h = pd.to_datetime(lt.EPOCH_SECONDS, unit='s').dt.hour
work_local, work_utc = lt.lh.between(7, 19).mean(), utc_h.between(7, 19).mean()
vt, _ = cramers_v(lt.lh, lt.ORDERING_TYPE)
add('R-BC2', 'Business/consumer', "Timestamps follow the sender's local working day (same for B and C)", 'share 07:00-19:59 local vs UTC; V(hour, sender type)',
    f'{work_local:.1%} local vs {work_utc:.1%} UTC; V = {vt:.3f}', 'Strong (time zone is approximate)', 'bc2_local_hour_profile.png',
    'Payments are entered and batched in office hours')
act = sql("""SELECT ORDERING_TYPE t, count(*)::DOUBLE / count(DISTINCT (ORDERING_BIC8, ORDERING_ACCOUNT)) per_acct FROM p GROUP BY 1""").set_index('t').per_acct
add('R-BC3', 'Business/consumer', 'Business sender accounts send more payments than consumer accounts', 'payments per (sender BIC, account) over the data window',
    f"B {act['B']:.1f} vs C {act['C']:.1f} (x{act['B']/act['C']:.1f})", 'Confirmed', 'bc3_activity_per_account.png',
    'Companies pay many suppliers and staff')
fr = sql("""SELECT anomaly, count(*) FILTER (WHERE ORDERING_TYPE='B') b, count(*) FILTER (WHERE ORDERING_TYPE='C') c
            FROM p WHERE anomaly IN ('CEO_SCAM','ROMANCE_SCAM') GROUP BY 1""").set_index('anomaly')
pure = (fr.loc['CEO_SCAM','c'] == 0) and (fr.loc['ROMANCE_SCAM','b'] == 0)
add('R-BC4', 'Business/consumer', 'CEO scams only come from business senders; romance scams only from consumer senders', 'labelled anomalies by sender type (full data)',
    f"CEO: B {fr.loc['CEO_SCAM','b']}, C {fr.loc['CEO_SCAM','c']}; romance: B {fr.loc['ROMANCE_SCAM','b']}, C {fr.loc['ROMANCE_SCAM','c']}",
    'Confirmed' if pure else 'Partial', 'bc4_anomaly_by_pair.csv', 'Each scam targets a specific kind of victim')

# ---- Rejected hypotheses (negative results)
vow = pd.Series(resid).groupby(n.CURRENCY_CODE.str[0].isin(list('AEIOU')).values).mean()
add('N1', 'Amount', 'Handout example: consonant currency codes give 3x amounts', 'ln residual difference consonant - vowel (3x would be +1.099)',
    f'{vow[False]-vow[True]:+.3f}', 'Rejected', '-', 'The handout example is illustrative, not a rule in this dataset')
n['xb'] = (n.o_cc != n.b_cc).astype(float); Wx = within(n, ['la', 'ob', 'bb', 'xb'])
bx = np.linalg.lstsq(Wx[['ob','bb','xb']].values, Wx.la.values, rcond=None)[0][2]
raw = n.groupby('xb').USD_AMOUNT.median()
add('N2', 'Amount', 'Cross-border payments are smaller than domestic ones', 'raw median ratio vs within-country ln effect',
    f'raw x{raw[1]/raw[0]:.2f}; same country and party types x{np.exp(bx):.2f}', 'Rejected (confounder: country mix)', 'amt4_crossborder_confounder.png',
    'Domestic payments come from richer countries; crossing a border itself changes nothing')
nl = pd.Series(resid).groupby((n.CURRENCY_CODE == n.o_cc.map(loc)).values).mean()
add('N3', 'Amount', 'Paying in a non-local currency changes the amount', 'ln residual difference non-local - local', f'{nl[False]-nl[True]:+.3f}', 'Rejected', '-', '-')
h = pd.to_datetime(n.EPOCH_SECONDS, unit='s')
add('N4', 'Amount', 'Transaction type, hour or weekday changes the amount', 'R² of ln amount',
    f"type {r2_by(n.la, n.TRANSACTION_TYPE_CODE):.4f}, hour {r2_by(n.la, h.dt.hour):.4f}, weekday {r2_by(n.la, h.dt.dayofweek):.4f}", 'Rejected', 'amt0_factor_screening.png', '-')

rules04 = pd.DataFrame(log, columns=RULE_COLS); rules04.to_csv(RULES / 'rules_04_core.csv', index=False)
rules04

,id,group,rule,test,value,verdict,evidence_files,interpretation
0,R-CCY1,Currency,Payments are made in the sender country's loca...,full data; local currency inferred in-sample,89.4% sender-local; only 4.2% use the receiver...,Partial (exceptions not yet explained),ccy1_currency_choice.png,The sender's bank debits in its home currency;...
1,R-CCY2,Currency,Direct payments: 1 conversion if sender and re...,prediction accuracy on all Direct rows,97.8%,Strong,ccy2_direct_conversion_rule.png,"With only two banks, FX happens once, and only..."
2,R-CCY3,Currency,Conversions never exceed the number of hops (p...,share of all rows,100.00%,Confirmed,ccy3_conversions_by_hops.csv,Each correspondent bank can convert at most once
3,R-AMT1,Amount,Amounts are log-normal: ln(amount) is normal w...,skew / excess kurtosis / sd of ln residual,"skew -0.016, kurtosis 0.024, sd 2.08",Confirmed,amt1_lognormal_by_pair.png,Payment sizes built from many multiplying fact...
4,R-AMT2,Amount,Each business party (sender or receiver) multi...,"ln effect within sender country, 95% cluster-b...","sender 1.999 [1.980, 2.019]; receiver 2.002 [1...",Confirmed,amt2_business_multiplier.png,Businesses pay invoices and payroll; consumers...
5,R-AMT3,Amount,The sender's country sets the base amount; the...,R² sender country vs extra R² of receiver country,0.131 vs 0.0003,Confirmed,amt3_country_base_amount.png,Payment size scales with the income level of t...
6,R-BC1,Business/consumer,Sender type and receiver type are assigned ind...,"chi-square p, Cramér's V","p = 0.67, V = 0.0003",Confirmed,bc1_type_independence.png,Who pays and who gets paid are separate draws
7,R-BC2,Business/consumer,Timestamps follow the sender's local working d...,"share 07:00-19:59 local vs UTC; V(hour, sender...",74.5% local vs 65.0% UTC; V = 0.024,Strong (time zone is approximate),bc2_local_hour_profile.png,Payments are entered and batched in office hours
8,R-BC3,Business/consumer,Business sender accounts send more payments th...,"payments per (sender BIC, account) over the da...",B 44.8 vs C 9.9 (x4.5),Confirmed,bc3_activity_per_account.png,Companies pay many suppliers and staff
9,R-BC4,Business/consumer,CEO scams only come from business senders; rom...,labelled anomalies by sender type (full data),"CEO: B 945, C 0; romance: B 0, C 2476",Confirmed,bc4_anomaly_by_pair.csv,Each scam targets a specific kind of victim


**Limits of these tests (short, for the methodology slide)**
- These tests show the rules hold in the supplied data. They can't reveal the generator's code itself.
- The time zone is approximate: multi-zone countries use their first listed zone.
- Local currency is inferred from the same payments it is tested on.

In [13]:
# Combined rules log for the whole team (re-run this cell after any notebook adds rules)
combine_rules()

,id,group,rule,test,value,verdict,evidence_files,interpretation,source,when_condition,then_effect,size,scope_rows_pct,exceptions,rejected_alternative
0,A2-example,Anomalies,Payment labelled DUPLICATE_PAYMENT and sender ...,NaN,median gap = ?? s,Confirmed / Strong / Partial / Rejected,a_duplicates_vs_control.csv; A2_duplicate_gap.png,INTERPRETATION: back-office race condition re-...,rules_03_anomalies.csv,Payment labelled DUPLICATE_PAYMENT and sender ...,"An earlier payment exists with same sender, be...",median gap = ?? s,?? of ?? duplicates (??%),C senders: different sender account (partner) ...,Exact copy of the row (rejected: timestamps di...
1,R-CCY1,Currency,Payments are made in the sender country's loca...,full data; local currency inferred in-sample,89.4% sender-local; only 4.2% use the receiver...,Partial (exceptions not yet explained),ccy1_currency_choice.png,The sender's bank debits in its home currency;...,rules_04_core.csv,NaN,NaN,NaN,NaN,NaN,NaN
2,R-CCY2,Currency,Direct payments: 1 conversion if sender and re...,prediction accuracy on all Direct rows,97.8%,Strong,ccy2_direct_conversion_rule.png,"With only two banks, FX happens once, and only...",rules_04_core.csv,NaN,NaN,NaN,NaN,NaN,NaN
3,R-CCY3,Currency,Conversions never exceed the number of hops (p...,share of all rows,100.00%,Confirmed,ccy3_conversions_by_hops.csv,Each correspondent bank can convert at most once,rules_04_core.csv,NaN,NaN,NaN,NaN,NaN,NaN
4,R-AMT1,Amount,Amounts are log-normal: ln(amount) is normal w...,skew / excess kurtosis / sd of ln residual,"skew -0.016, kurtosis 0.024, sd 2.08",Confirmed,amt1_lognormal_by_pair.png,Payment sizes built from many multiplying fact...,rules_04_core.csv,NaN,NaN,NaN,NaN,NaN,NaN
5,R-AMT2,Amount,Each business party (sender or receiver) multi...,"ln effect within sender country, 95% cluster-b...","sender 1.999 [1.980, 2.019]; receiver 2.002 [1...",Confirmed,amt2_business_multiplier.png,Businesses pay invoices and payroll; consumers...,rules_04_core.csv,NaN,NaN,NaN,NaN,NaN,NaN
6,R-AMT3,Amount,The sender's country sets the base amount; the...,R² sender country vs extra R² of receiver country,0.131 vs 0.0003,Confirmed,amt3_country_base_amount.png,Payment size scales with the income level of t...,rules_04_core.csv,NaN,NaN,NaN,NaN,NaN,NaN
7,R-BC1,Business/consumer,Sender type and receiver type are assigned ind...,"chi-square p, Cramér's V","p = 0.67, V = 0.0003",Confirmed,bc1_type_independence.png,Who pays and who gets paid are separate draws,rules_04_core.csv,NaN,NaN,NaN,NaN,NaN,NaN
8,R-BC2,Business/consumer,Timestamps follow the sender's local working d...,"share 07:00-19:59 local vs UTC; V(hour, sender...",74.5% local vs 65.0% UTC; V = 0.024,Strong (time zone is approximate),bc2_local_hour_profile.png,Payments are entered and batched in office hours,rules_04_core.csv,NaN,NaN,NaN,NaN,NaN,NaN
9,R-BC3,Business/consumer,Business sender accounts send more payments th...,"payments per (sender BIC, account) over the da...",B 44.8 vs C 9.9 (x4.5),Confirmed,bc3_activity_per_account.png,Companies pay many suppliers and staff,rules_04_core.csv,NaN,NaN,NaN,NaN,NaN,NaN


## 4. Files for teammates
- `data/processed/payments_clean.parquet`: all 10M rows with the shared derived columns (`rid, ts, o_cc, b_cc, pair, anomaly, cross_border, path_len, path_gap, o_det_cc, b_det_cc, o_det_acct, b_det_acct, b_party_text`). Query it with DuckDB. Don't load all rows into pandas.
- `data/processed/sample_20pct.parquet`: for charts and models. It **still contains anomalies**, so filter `anomaly == 'NONE'` for normal-only work.
- `data/processed/anomalies.parquet`: every labelled anomaly with all text columns.
- Put your rules in `outputs/tables/rules/rules_<notebook>.csv` and run `combine_rules()`, which updates `outputs/tables/rules_log.csv`.
- If the derived columns change, bump `SCHEMA_VERSION` in the setup cell (all notebooks), and this notebook rebuilds the files.